Cell 1 — install:

In [ ]:
!pip install -U sentence-transformers -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 11.2 MB/s eta 0:00:00


Cell 2 — upload catalog + load embedding model:

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
from sentence_transformers import SentenceTransformer
import numpy as np

catalog_df = pd.read_json('/content/drive/MyDrive/ai-interior-designer/furniture_catalog.json')

embed_model = SentenceTransformer("all-MiniLM-L6-v2")

catalog_df["search_text"] = (
    catalog_df["name"] + " " + catalog_df["category"] + " " + catalog_df["style"] + " style furniture"
)
catalog_embeddings = embed_model.encode(catalog_df["search_text"].tolist())

print(f"Embedded {len(catalog_df)} items")

Mounted at /content/drive


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedded 3694 items



Cell 3 — retrieval function:

In [ ]:
def retrieve_furniture(query: str, top_k: int = 5, max_price: float = None) -> pd.DataFrame:
    query_embedding = embed_model.encode([query])
    similarities = np.dot(catalog_embeddings, query_embedding.T).flatten()

    results = catalog_df.copy()
    results["similarity"] = similarities

    if max_price is not None:
        results = results[results["price_usd"] <= max_price]

    return results.sort_values("similarity", ascending=False).head(top_k)

Cell 4 — test it:

In [ ]:
catalog_df = catalog_df.drop_duplicates(subset=["name", "category", "price_usd"]).reset_index(drop=True)

catalog_df["search_text"] = (
    catalog_df["name"] + " " + catalog_df["category"] + " " + catalog_df["style"] + " style furniture"
)
catalog_embeddings = embed_model.encode(catalog_df["search_text"].tolist())

print(f"Embedded {len(catalog_df)} items after deduplication")

Embedded 3217 items after deduplication


In [ ]:
retrieved = retrieve_furniture("modern minimalist bedroom furniture", top_k=5, max_price=500)
print(retrieved[["name", "category", "style", "price_usd", "similarity"]])

         name category         style  price_usd  similarity
37     MINNEN     Beds  contemporary     106.65    0.724186
61   TYSSEDAL     Beds  contemporary     376.65    0.699227
80      MYDAL     Beds  contemporary     241.65    0.697771
114     MYDAL     Beds  contemporary     214.65    0.697771
57    SNIGLAR     Beds  contemporary      52.65    0.677940


Cell 5 — extend Step 2's plan_design to accept retrieved furniture as context:

In [ ]:
CATEGORY_MAP = {
    "bed": "Beds",
    "table": "Tables & desks",
    "nightstand": "Chests of drawers & drawer units",
    "wardrobe": "Wardrobes",
    "chair": "Chairs",
    "sofa": "Sofas & armchairs",
    "bookcase": "Bookcases & shelving units",
}

def get_relevant_categories(furniture_list):
    categories = []
    for item in furniture_list:
        item_lower = item.lower()
        for key, cat in CATEGORY_MAP.items():
            if key in item_lower and cat not in categories:
                categories.append(cat)
    return categories


def plan_design_with_rag(room, style, budget, preferred_colors, budget_num):
    relevant_categories = get_relevant_categories(room.furniture + ["wardrobe"])
    all_retrieved = []
    for category in relevant_categories:
        subset = catalog_df[catalog_df["category"] == category]
        if subset.empty:
            continue
        query_embedding = embed_model.encode([style + " " + category])
        subset_embeddings = catalog_embeddings[subset.index]
        similarities = np.dot(subset_embeddings, query_embedding.T).flatten()
        subset = subset.copy()
        subset["similarity"] = similarities
        subset = subset[subset["price_usd"] <= budget_num].sort_values("similarity", ascending=False).head(3)
        all_retrieved.append(subset)

    if all_retrieved:
        retrieved = pd.concat(all_retrieved).drop_duplicates(subset=["name", "category", "price_usd"])
    else:
        retrieved = pd.DataFrame()

    lines = []
    for _, row in retrieved.iterrows():
        line = "- " + row["name"] + " (" + row["category"] + ", " + row["style"] + "): $" + str(row["price_usd"])
        lines.append(line)
    furniture_options = "\n".join(lines)

    header = "You are an interior design planner. Given a room analysis, user preferences, and a list of ACTUAL AVAILABLE furniture items, create a design plan using ONLY items from the provided list. If lighting is needed but not in the list, note it as not available in catalog rather than inventing an item."

    room_json = room.model_dump_json(indent=2)
    colors_str = ", ".join(preferred_colors)

    schema = """{
  "design_style": "string",
  "color_palette": ["color1", "color2"],
  "furniture_changes": [
    {"item": "exact name from available list, or not available in catalog", "description": "string", "action": "replace/add/keep", "price_usd": 0}
  ],
  "layout_changes": ["string"],
  "generation_prompt": "string describing the redesigned room for an image generation model"
}"""

    prompt = header + "\n\nRoom analysis:\n" + room_json + "\n\nUser preferences:\n- Style: " + style + "\n- Budget: " + budget + "\n- Preferred colors: " + colors_str + "\n\nAvailable furniture (choose only from this list):\n" + furniture_options + "\n\nRespond with ONLY a valid JSON object, no other text, in exactly this format:\n\n" + schema + "\n\nDo not include any explanation, markdown formatting, or text outside the JSON object."

    return prompt

Cell 6 — test what the constructed prompt actually looks like, before spending an API call on it:

In [ ]:
from pydantic import BaseModel

class RoomAnalysis(BaseModel):
    room_type: str
    furniture: list[str]
    decor: list[str]
    colors: list[str]
    style: str
    lighting: str
    spatial_observations: list[str]

room = RoomAnalysis(
    room_type="Bedroom",
    furniture=["bed", "table", "nightstand"],
    decor=["paintings", "wooden wardrobe"],
    colors=["beige", "brown"],
    style="Traditional",
    lighting="table lamps",
    spatial_observations=["bed against the wall", "nightstand on the left side"]
)

print(room)

room_type='Bedroom' furniture=['bed', 'table', 'nightstand'] decor=['paintings', 'wooden wardrobe'] colors=['beige', 'brown'] style='Traditional' lighting='table lamps' spatial_observations=['bed against the wall', 'nightstand on the left side']


In [ ]:
print(catalog_df["category"].unique())

['Bar furniture' 'Beds' 'Bookcases & shelving units'
 'Cabinets & cupboards' 'Café furniture' 'Chairs'
 'Chests of drawers & drawer units' "Children's furniture"
 'Nursery furniture' 'Outdoor furniture' 'Room dividers'
 'Sideboards, buffets & console tables' 'Sofas & armchairs'
 'Tables & desks' 'Trolleys' 'TV & media furniture' 'Wardrobes']


In [ ]:
test_prompt = plan_design_with_rag(
    room=room,
    style="modern minimalist",
    budget="$1500",
    preferred_colors=["beige", "white"],
    budget_num=1500
)
print(test_prompt)

You are an interior design planner. Given a room analysis, user preferences, and a list of ACTUAL AVAILABLE furniture items, create a design plan using ONLY items from the provided list. If lighting is needed but not in the list, note it as not available in catalog rather than inventing an item.

Room analysis:
{
  "room_type": "Bedroom",
  "furniture": [
    "bed",
    "table",
    "nightstand"
  ],
  "decor": [
    "paintings",
    "wooden wardrobe"
  ],
  "colors": [
    "beige",
    "brown"
  ],
  "style": "Traditional",
  "lighting": "table lamps",
  "spatial_observations": [
    "bed against the wall",
    "nightstand on the left side"
  ]
}

User preferences:
- Style: modern minimalist
- Budget: $1500
- Preferred colors: beige, white

Available furniture (choose only from this list):
- MINNEN (Beds, contemporary): $106.65
- TYSSEDAL (Beds, contemporary): $376.65
- MYDAL (Beds, contemporary): $241.65
- LACK (Tables & desks, modern): $8.1
- LACK (Tables & desks, modern): $24.03
- 

Cell 7 — actual Gemini call + validation, reusing your existing DesignPlan/FurnitureItem schema and parse_design_plan (paste those in if this notebook doesn't have them yet — same pattern as notebook 2):

In [ ]:
import time

def call_gemini_with_retry(prompt: str, primary_model="gemini-3.6-flash", fallback_model="gemini-2.5-flash-lite", max_retries=4):
    for attempt in range(max_retries):
        try:
            response = client.models.generate_content(model=primary_model, contents=prompt)
            return response
        except Exception as e:
            print(f"Attempt {attempt+1} with {primary_model} failed: {e}")
            time.sleep(3 * (attempt + 1))

    print(f"Primary model failed after {max_retries} attempts, trying fallback: {fallback_model}")
    response = client.models.generate_content(model=fallback_model, contents=prompt)
    return response

response = call_gemini_with_retry(test_prompt)
print("RAW:", repr(response.text))
design = parse_design_plan(response.text)
print("PARSED:", design)

RAW: '{\n  "design_style": "Modern Minimalist",\n  "color_palette": [\n    "white",\n    "beige"\n  ],\n  "furniture_changes": [\n    {\n      "item": "TYSSEDAL",\n      "description": "White contemporary bed frame replacing the traditional bed to establish a clean, minimalist focal point.",\n      "action": "replace",\n      "price_usd": 376.65\n    },\n    {\n      "item": "TYSSEDAL",\n      "description": "White contemporary wardrobe replacing the heavy wooden wardrobe for cohesive modern storage.",\n      "action": "replace",\n      "price_usd": 444.15\n    },\n    {\n      "item": "LACK",\n      "description": "Simple modern white table replacing the bulky traditional nightstand.",\n      "action": "replace",\n      "price_usd": 24.03\n    },\n    {\n      "item": "TYSSEDAL",\n      "description": "White chest of drawers added to provide additional sleek storage while keeping clutter hidden.",\n      "action": "add",\n      "price_usd": 255.15\n    },\n    {\n      "item": "not av